In [15]:
import pandas as pd

df = pd.read_csv("ml_dataset.csv")
df["delivery_day"] = pd.to_datetime(df["delivery_day"])

df = df[df["delivery_day"] >= pd.Timestamp("2024-01-01")]
print(df.shape)
print(df.head())

(147464, 26)
       zone_id zone_code delivery_day  hour  price_value  buy_volume_value  \
78906       13       NO2   2024-01-01     0        53.79            4861.8   
78907       13       NO2   2024-01-01     1        48.98            4613.5   
78908       13       NO2   2024-01-01     2        27.45            4210.6   
78909       13       NO2   2024-01-01     3        24.48            4328.0   
78910       13       NO2   2024-01-01     4        24.01            4329.4   

       sell_volume_value  net_volume  incoming_flow  outgoing_flow  ...  \
78906             3309.4      1552.4         3731.7         1930.7  ...   
78907             3143.6      1469.9         3731.7         2016.3  ...   
78908             2698.4      1512.2         3711.3         1955.7  ...   
78909             3285.0      1043.0         3731.7         2448.3  ...   
78910             3683.7       645.7         3731.7         2894.7  ...   

                  datetime  price_lag_1  price_lag_2  price_lag_3  

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score


drop_cols = [
    "delivery_day",
    "datetime",
    "zone_code"
]

X = df.drop(columns=drop_cols + ["target_price_next_hour"])
y = df["target_price_next_hour"]

print("Features:", len(X.columns))

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
    

Features: 22
Train rows: 117,971
Test rows : 29,493
Training...
Predicting...


In [ ]:
from logging import root


rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

for lag in [1, 24, 48, 168]:
    rmse = root_mean_squared_error(
    y_test,
    X_test[f"price_lag_{lag}"]
    )
    print(f"lag_{lag}: {rmse:.4f}")



MAE = 6.0283
lag_1: 5.7438
lag_24: 23.2629
lag_48: 35.8982
lag_168: 54.7821
RMSE: 54.782088721267606
R²: 0.9785721451970495


In [19]:
importance = (
pd.DataFrame({
"feature": X.columns,
"importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
)
print(importance.head(20))

              feature  importance
2         price_value    0.978162
14        price_lag_1    0.002503
16        price_lag_3    0.002340
17       price_lag_24    0.002293
20        price_ma_24    0.002266
4   sell_volume_value    0.001547
15        price_lag_2    0.001366
21       price_ma_168    0.001322
3    buy_volume_value    0.001233
5          net_volume    0.001122
18       price_lag_48    0.001016
19      price_lag_168    0.000997
13        day_of_year    0.000797
1                hour    0.000656
7       outgoing_flow    0.000596
6       incoming_flow    0.000502
8            net_flow    0.000489
9     import_capacity    0.000242
11        day_of_week    0.000239
12              month    0.000220
